# OSFI Regulation Assistant: a RAG project
## Part 2: Find near-duplicates, build embeddings, and search

Part 1 produced 186 clean chunks. Part 2 builds the **retrieval** half of RAG: given a question, find the passages most likely to contain the answer.

**What this notebook covers:**
1. **Near-duplicate detection with MinHash**, since guidelines and their infosheets often repeat passages
2. **Embeddings and cosine similarity**, turning text into vectors so passages with similar *meaning* are close together
3. **Keyword search with BM25**, the classic search-engine method that excels at exact terms like "GDS" or "HELOC"
4. **Hybrid search**, combining both rankings with Reciprocal Rank Fusion
5. **Re-ranking with a cross-encoder**, a slower but more precise second pass
6. **A side-by-side comparison** of all four methods on sample questions

**The search code lives in a Python module (`retriever.py`), not just in notebook cells.** It's a small, reusable class that Part 3 and the API in Part 5 will import. Writing it this way is the step from notebook analysis toward production-quality code.

**Runtime:** a **T4 GPU** is fastest, but a CPU works too, since 186 chunks embed in under a minute.

## Step 1: Install libraries and load the chunks from Part 1

In [ ]:
!pip -q install sentence-transformers rank_bm25 faiss-cpu datasketch nltk

import os, sys, json, time
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 200)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/osfi_rag_project"
except Exception as e:
    print("Drive not available, using a local folder:", e)
    PROJECT_DIR = "osfi_rag_project"

DATA_DIR = os.path.join(PROJECT_DIR, "data")
INDEX_DIR = os.path.join(PROJECT_DIR, "index")
CODE_DIR = os.path.join(PROJECT_DIR, "osfi_rag")
os.makedirs(INDEX_DIR, exist_ok=True); os.makedirs(CODE_DIR, exist_ok=True)

chunks_df = pd.read_json(os.path.join(DATA_DIR, "chunks.jsonl"), lines=True)
print(f"{len(chunks_df)} chunks loaded")
chunks_df.groupby("doc_id", sort=False).size().rename("chunks").to_frame().T

## Step 2: Find near-duplicate passages with MinHash

Exact duplicates were removed in Part 1, but **near-duplicates** remain: the same passage with a word or two changed, or a guideline sentence quoted in an infosheet. If both copies stay, search returns the same content twice and pushes out other useful passages.

**How MinHash works, in brief:** each chunk is broken into overlapping 5-word phrases ("shingles"). Two chunks that share most of their shingles are near-duplicates, measured by **Jaccard similarity** (shared shingles ÷ all shingles). Comparing every pair exactly gets slow as collections grow, so MinHash compresses each chunk into a short "signature", and **LSH** (locality-sensitive hashing) finds likely matches without comparing every pair. It's the same technique search engines use to find duplicate web pages.

MinHash only *estimates* similarity, so we use it in two stages: a looser threshold (0.7) to flag **candidates**, then the exact Jaccard similarity to **confirm** (0.8) before removing anything. Without this, pairs right at the threshold can be missed.

In [ ]:
from datasketch import MinHash, MinHashLSH
import re

THRESHOLD = 0.8         # exact Jaccard similarity at which two chunks count as near-duplicates
LSH_THRESHOLD = 0.7     # looser, so MinHash's estimate doesn't miss pairs near the line

def shingles(text, n=5):
    words = re.findall(r"\w+", text.lower())
    return {" ".join(words[i:i + n]) for i in range(max(1, len(words) - n + 1))}

signatures, shingle_sets = {}, {}
lsh = MinHashLSH(threshold=LSH_THRESHOLD, num_perm=128)
for row in chunks_df.itertuples():
    s = shingles(row.text)
    m = MinHash(num_perm=128)
    for sh in s:
        m.update(sh.encode("utf-8"))
    signatures[row.chunk_id], shingle_sets[row.chunk_id] = m, s
    lsh.insert(row.chunk_id, m)

pairs = set()
for cid, m in signatures.items():
    for other in lsh.query(m):
        if other != cid:
            pairs.add(tuple(sorted((cid, other))))

def jaccard(a, b):
    return len(shingle_sets[a] & shingle_sets[b]) / len(shingle_sets[a] | shingle_sets[b])

text_of = chunks_df.set_index("chunk_id")["text"]
dupes = pd.DataFrame([{"chunk_a": a, "chunk_b": b, "jaccard": round(jaccard(a, b), 3),
                       "text_a": text_of[a][:90], "text_b": text_of[b][:90]} for a, b in sorted(pairs)])
print(f"MinHash flagged {len(dupes)} candidate pairs. "
      f"{(dupes['jaccard'] >= THRESHOLD).sum() if len(dupes) else 0} confirmed with exact Jaccard >= {THRESHOLD}")
dupes.sort_values("jaccard", ascending=False).head(15) if len(dupes) else dupes

**Which copy to keep?** When two chunks are near-duplicates, we keep the one from the **more authoritative document**: the guideline itself over an infosheet or clarification. Within the same document, we keep the earlier chunk. The check below uses the *exact* Jaccard similarity, so MinHash's estimate is confirmed before anything is removed.

In [ ]:
PRIORITY = {"B-20": 0, "B-21": 1, "E-23": 2, "MI-NOTE": 3, "B-20-CLAR": 4, "B-20-INFO": 5}
doc_of = chunks_df.set_index("chunk_id")["doc_id"]

to_drop = set()
for row in dupes.itertuples() if len(dupes) else []:
    if row.jaccard < THRESHOLD:
        continue                                  # MinHash estimate was too generous; keep both
    a, b = row.chunk_a, row.chunk_b
    if a in to_drop or b in to_drop:
        continue
    keep_a = (PRIORITY.get(doc_of[a], 9), a) <= (PRIORITY.get(doc_of[b], 9), b)
    to_drop.add(b if keep_a else a)

clean_df = chunks_df[~chunks_df["chunk_id"].isin(to_drop)].reset_index(drop=True)
print(f"Removed {len(to_drop)} near-duplicate chunks: {len(chunks_df)} -> {len(clean_df)} chunks")

## Step 3: Save the retrieval code as a reusable module

The next cell writes **`retriever.py`** into your Drive project folder. It contains one class, `HybridRetriever`, with a method for each search approach. Read through it once: the comments explain each part, and this file will appear in your GitHub repository.

`%%writefile` is a notebook command that saves a cell's contents to a file instead of running it.

In [ ]:
%%writefile {CODE_DIR}/retriever.py
"""Hybrid retrieval over OSFI guideline chunks (version 2).

Combines keyword search (BM25) with semantic search (embeddings + cosine
similarity), merges the two rankings with Reciprocal Rank Fusion, and can
re-rank the merged candidates with a cross-encoder.

Version 2 changes, from the Part 4 evaluation:
- Keyword search stems words, so "define", "defines" and "definition" match.
- The re-ranker sees 50 candidates instead of 20, since it can only promote
  passages that the first stage retrieved.
"""
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass, asdict

import faiss
import numpy as np
from nltk.stem import PorterStemmer
from rank_bm25 import BM25Okapi

STOPWORDS = set("""a an and are as at be by can do does for from has have how i in is it its
of on or should that the their them these this to under was what when which who why will
with""".split())

# BGE embedding models work best when search queries carry this instruction
QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "


_stemmer = PorterStemmer()


def tokenize(text: str, stem: bool = True) -> list[str]:
    """Lower-case words for BM25, keeping acronyms like GDS and numbers like 80.
    With stem=True, words are reduced to their root: define/defines/defining -> defin."""
    words = [w for w in re.findall(r"[a-z0-9]+(?:[-'][a-z0-9]+)*", text.lower()) if w not in STOPWORDS]
    return [_stemmer.stem(w) for w in words] if stem else words


@dataclass
class SearchResult:
    rank: int
    score: float
    chunk_id: str
    doc_id: str
    section: str
    text: str
    url: str

    def to_dict(self) -> dict:
        return asdict(self)


class HybridRetriever:
    """Keyword + semantic search over a list of chunk dictionaries."""

    def __init__(self, chunks: list[dict], embedder, reranker=None, rrf_k: int = 60,
                 stem: bool = True, rerank_candidates: int = 50):
        self.chunks = chunks
        self.embedder = embedder
        self.reranker = reranker
        self.rrf_k = rrf_k
        self.stem = stem
        self.rerank_candidates = rerank_candidates
        self.bm25 = BM25Okapi([tokenize(c["embed_text"], stem) for c in chunks])
        self.index = None

    # ---------- building and saving the vector index ----------
    def build_index(self, batch_size: int = 32) -> np.ndarray:
        vectors = self.embedder.encode([c["embed_text"] for c in self.chunks], batch_size=batch_size,
                                       normalize_embeddings=True, show_progress_bar=True)
        self._set_vectors(np.asarray(vectors, dtype="float32"))
        return self.vectors

    def _set_vectors(self, vectors: np.ndarray) -> None:
        self.vectors = vectors
        # Inner product on unit-length vectors equals cosine similarity
        self.index = faiss.IndexFlatIP(vectors.shape[1])
        self.index.add(vectors)

    def save(self, folder: str) -> None:
        os.makedirs(folder, exist_ok=True)
        np.save(os.path.join(folder, "embeddings.npy"), self.vectors)
        with open(os.path.join(folder, "chunks.jsonl"), "w", encoding="utf-8") as f:
            for c in self.chunks:
                f.write(json.dumps(c, ensure_ascii=False) + "\n")

    @classmethod
    def load(cls, folder: str, embedder, reranker=None, **kwargs) -> "HybridRetriever":
        with open(os.path.join(folder, "chunks.jsonl"), encoding="utf-8") as f:
            chunks = [json.loads(line) for line in f]
        retriever = cls(chunks, embedder, reranker, **kwargs)
        retriever._set_vectors(np.load(os.path.join(folder, "embeddings.npy")))
        return retriever

    # ---------- individual search methods (return (chunk index, score) pairs) ----------
    def keyword_search(self, query: str, k: int = 20) -> list[tuple[int, float]]:
        scores = self.bm25.get_scores(tokenize(query, self.stem))
        top = np.argsort(scores)[::-1][:k]
        return [(int(i), float(scores[i])) for i in top if scores[i] > 0]

    def semantic_search(self, query: str, k: int = 20) -> list[tuple[int, float]]:
        q = self.embedder.encode([QUERY_INSTRUCTION + query], normalize_embeddings=True)
        scores, ids = self.index.search(np.asarray(q, dtype="float32"), k)
        return [(int(i), float(s)) for i, s in zip(ids[0], scores[0]) if i != -1]

    def hybrid_search(self, query: str, k: int = 20, candidates: int | None = None) -> list[tuple[int, float]]:
        """Reciprocal Rank Fusion: reward chunks that rank well in either list."""
        candidates = candidates or max(30, k)
        fused: dict[int, float] = {}
        for ranking in (self.keyword_search(query, candidates), self.semantic_search(query, candidates)):
            for rank, (i, _) in enumerate(ranking, start=1):
                fused[i] = fused.get(i, 0.0) + 1.0 / (self.rrf_k + rank)
        return sorted(fused.items(), key=lambda x: x[1], reverse=True)[:k]

    def rerank(self, query: str, hits: list[tuple[int, float]], k: int) -> list[tuple[int, float]]:
        """A cross-encoder reads the question and each passage together, which is slower but more precise."""
        if self.reranker is None or not hits:
            return hits[:k]
        scores = self.reranker.predict([(query, self.chunks[i]["embed_text"]) for i, _ in hits])
        order = np.argsort(scores)[::-1][:k]
        return [(hits[j][0], float(scores[j])) for j in order]

    # ---------- the main entry point ----------
    def search(self, query: str, k: int = 5, method: str = "hybrid_rerank") -> list[SearchResult]:
        if method == "keyword":
            hits = self.keyword_search(query, k)
        elif method == "semantic":
            hits = self.semantic_search(query, k)
        elif method == "hybrid":
            hits = self.hybrid_search(query, k)
        elif method == "hybrid_rerank":
            hits = self.rerank(query, self.hybrid_search(query, self.rerank_candidates), k)
        else:
            raise ValueError(f"Unknown method: {method}")
        return [SearchResult(rank=r, score=round(s, 4), chunk_id=self.chunks[i]["chunk_id"],
                             doc_id=self.chunks[i]["doc_id"], section=self.chunks[i]["section"],
                             text=self.chunks[i]["text"], url=self.chunks[i]["url"])
                for r, (i, s) in enumerate(hits, start=1)]

In [ ]:
sys.path.insert(0, PROJECT_DIR)
import importlib, osfi_rag.retriever as retriever_module
importlib.reload(retriever_module)
from osfi_rag.retriever import HybridRetriever, tokenize
open(os.path.join(CODE_DIR, "__init__.py"), "a").close()
print("Module loaded. Example tokenization:", tokenize("What is the GDS ratio under Guideline B-20?"))

## Step 4: Load the embedding and re-ranking models

- **Embedding model: `BAAI/bge-small-en-v1.5`.** A small, open-source model that turns each passage into **384 numbers**. Passages with similar meaning get similar vectors. It's compact enough to run quickly on a free GPU or even a CPU, and it ranks well on public retrieval benchmarks for its size.
- **Re-ranker: `cross-encoder/ms-marco-MiniLM-L-6-v2`.** Embedding search compares a question and a passage that were encoded *separately*. A cross-encoder reads them *together*, so it judges relevance more precisely, but it's too slow to run on every passage. So we only use it on the top 20 candidates.

Both download from Hugging Face the first time (a few hundred MB in total).

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
reranker = CrossEncoder(RERANK_MODEL, device=DEVICE)
print("Running on:", DEVICE)

## Step 5: Build the index

Each chunk's `embed_text` (its document title and section header, plus the passage, from Part 1) is turned into a vector. The vectors are **normalized to length 1**, so the dot product between two vectors equals their **cosine similarity**: 1 means identical direction (same meaning), 0 means unrelated. **FAISS**, a vector search library from Meta, stores the vectors and finds the closest ones quickly.

In [ ]:
records = clean_df.to_dict("records")
retriever = HybridRetriever(records, embedder, reranker)
t0 = time.time()
vectors = retriever.build_index()
print(f"Embedded {vectors.shape[0]} chunks into {vectors.shape[1]}-dimensional vectors in {time.time() - t0:.1f}s")

### What cosine similarity looks like

For one question, the scores of the closest and furthest passages. Relevant passages typically score noticeably higher than unrelated ones, though exact values depend on the model.

In [ ]:
from osfi_rag.retriever import QUERY_INSTRUCTION
question = "What is the minimum qualifying rate for an uninsured mortgage?"
q_vec = embedder.encode([QUERY_INSTRUCTION + question], normalize_embeddings=True)[0]
sims = vectors @ q_vec                                  # cosine similarity with every chunk
order = np.argsort(sims)[::-1]
show = list(order[:3]) + list(order[-2:])
pd.DataFrame({"cosine_similarity": sims[show].round(3),
              "doc_id": clean_df.loc[show, "doc_id"].values,
              "section": clean_df.loc[show, "section"].str[:60].values,
              "text": clean_df.loc[show, "text"].str[:90].values},
             index=["closest 1", "closest 2", "closest 3", "furthest 2", "furthest 1"])

## Step 6: Compare the four search methods

Each sample question below has the document we **expect** the answer to come from. This is a quick sanity check, not a full evaluation: Part 4 builds a proper test set with precise metrics.

Watch for two patterns:
- **Keyword search** is strong when the question uses the document's exact terms (like "HELOC" or "GDS"), and weak when it uses different words for the same idea.
- **Semantic search** handles rephrased questions, but can miss exact codes and acronyms. Hybrid search aims to get the best of both.

In [ ]:
SAMPLE_QUESTIONS = [
    ("What is the minimum qualifying rate for uninsured mortgages?", "B-20"),
    ("What is the maximum loan-to-value ratio for a HELOC?", "B-20"),
    ("How should lenders calculate the GDS and TDS ratios?", "B-20"),
    ("What should mortgage insurers check when verifying a borrower's income?", "B-21"),
    ("How does OSFI define a model in its model risk guideline?", "E-23"),
    ("How should a bank work out the actual cost of mortgage insurance it charges borrowers?", "MI-NOTE"),
    ("Do the B-20 rules apply to combined loan plans and other innovative lending products?", "B-20-CLAR"),
    ("Why is high household debt a risk to Canada's financial system?", "B-20-INFO"),
]
METHODS = ["keyword", "semantic", "hybrid", "hybrid_rerank"]

rows, detail = [], []
for q, expected in SAMPLE_QUESTIONS:
    row = {"question": q[:60], "expected": expected}
    for m in METHODS:
        hits = retriever.search(q, k=3, method=m)
        row[m] = "✓" if any(h.doc_id == expected for h in hits) else "✗"
        row[m + "_top1"] = hits[0].doc_id if hits else "-"
        detail += [{"question": q[:50], "method": m, **{k: v for k, v in h.to_dict().items() if k != "url"}} for h in hits]
    rows.append(row)

comparison = pd.DataFrame(rows)
print("✓ = the expected document appears in the top 3 results")
comparison[["question", "expected"] + METHODS]

In [ ]:
hit_rate = {m: (comparison[m] == "✓").mean() for m in METHODS}
pd.Series(hit_rate, name="share of questions with the expected document in the top 3").map("{:.0%}".format).to_frame()

### Look closely at one question

In [ ]:
QUESTION = SAMPLE_QUESTIONS[1][0]   # change the number to inspect a different question
print("Q:", QUESTION, "\n")
for m in METHODS:
    print(f"--- {m} ---")
    for h in retriever.search(QUESTION, k=3, method=m):
        print(f"  {h.rank}. [{h.doc_id} | {h.section[:55]}] score={h.score}")
        print(f"     {h.text[:160]}")

## Step 7: Save the index

The cleaned chunks and their vectors are saved to Drive, so Part 3 loads them in seconds instead of rebuilding.

In [ ]:
retriever.save(INDEX_DIR)
json.dump({"embed_model": EMBED_MODEL, "rerank_model": RERANK_MODEL, "chunks": len(records),
           "near_duplicates_removed": len(to_drop)}, open(os.path.join(INDEX_DIR, "config.json"), "w"), indent=2)
pd.DataFrame(detail).to_csv(os.path.join(INDEX_DIR, "method_comparison_detail.csv"), index=False)
print("Saved to", INDEX_DIR, ":", sorted(os.listdir(INDEX_DIR)))

## Checkpoint

**Send me the outputs of:**
1. **Step 2:** the number of near-duplicate pairs (and the table if any were found), plus the "Removed ..." line
2. **Step 5:** the embedding line and the cosine-similarity table
3. **Step 6:** the comparison table, the hit-rate table, and the close-up of one question

Then we'll build Part 3: generating answers with an LLM, with citations.